## DYAMOND GEOS — temperature over time

Reads the NSF DYAMOND GEOS OpenVisus dataset for **temperature** (`T`), samples the timesteps you configure, and writes ParaView-friendly output.

**What to open in ParaView:** `temperature_over_time.pvd` (VTK collection). Per-timestep grids are `temperature_over_time_out/frame_XXXX.vti`.

Adjust `frames`, `face`, and `quality` in the params cell to match your analysis.

In [1]:
import numpy as np
import OpenVisus as ov
from pathlib import Path
import pyvista as pv
from tqdm import tqdm

In [2]:
# --- params: edit these ---
variable = "T"  # temperature field in GEOS DYAMOND idx
face = 2
quality = -8

num_frames = 500
time_max = 10268
frames = np.linspace(0, time_max, num_frames, dtype=int).tolist()

out_dir = Path("temperature_over_time_out")
pvd_path = Path("temperature_over_time.pvd")
scalar_name = "temperature"

out_dir.mkdir(parents=True, exist_ok=True)

In [3]:
idx_url = (
    f"https://nsdf-climate3-origin.nationalresearchplatform.org:50098/"
    f"nasa/nsdf/climate3/dyamond/GEOS/GEOS_{variable.upper()}/"
    f"{variable.lower()}_face_{face}_depth_52_time_0_10269.idx"
)
db = ov.LoadDataset(idx_url)

data = db.read(time=0, quality=quality)
Z, Y, X = data.shape
grid = pv.ImageData()
grid.dimensions = (X, Y, Z)
grid.origin = (0, 0, 0)
grid.spacing = (1, 1, 20)
print(f"Field {variable}, shape (Z,Y,X)= {data.shape}, idx: {idx_url}")

Field T, shape (Z,Y,X)= (7, 180, 360), idx: https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/GEOS/GEOS_T/t_face_2_depth_52_time_0_10269.idx


In [4]:
for i, t in enumerate(tqdm(frames, desc="temperature frames")):
    arr = db.read(time=int(t), quality=quality)
    flat = arr.flatten(order="C")
    grid[scalar_name] = flat
    grid.save(out_dir / f"frame_{i:04d}.vti")

print("Done writing VTI frames.")

temperature frames: 100%|████████████████████████████████████████████████████████████| 500/500 [05:36<00:00,  1.48it/s]

Done writing VTI frames.


In [5]:
rel = out_dir.name
with open(pvd_path, "w", encoding="utf-8") as f:
    f.write('<?xml version="1.0"?>\n')
    f.write('<VTKFile type="Collection" version="0.1">\n')
    f.write("  <Collection>\n")
    for i, t in enumerate(frames):
        fname = f"{rel}/frame_{i:04d}.vti"
        f.write(f'    <DataSet timestep="{float(t)}" file="{fname}"/>\n')
    f.write("  </Collection>\n")
    f.write("</VTKFile>\n")

print(f"Wrote {pvd_path.resolve()} — open this file in ParaView.")

Wrote C:\Users\tjhes\OneDrive\Desktop\VisScientificDataTeam\SciVis-Project\pressure_and_temp\temperature_over_time.pvd — open this file in ParaView.
